# 03 Feature Engineering Starter

Project: **AI Support Operations SLA Breach and Priority Triage**

## Objective
Create leakage-safe features for the SLA breach classifier and optional resolution-hours regressor.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

path = Path("../01_Data/processed data/cleaned_support_sla_sample.csv")
df = pd.read_csv(path, parse_dates=["submitted_at"], low_memory=False)
df.head()


,ticket_id,account_id,customer_segment,uk_region,support_channel,product_area,issue_category,priority_initial,submitted_at,day_of_week,...,assigned_agent,agent_experience_months,backlog_age_hours,first_response_minutes,reopened_last_90d,resolved_at,escalated_prior_to_resolution,resolution_hours,final_priority,sla_breached
0,TCK-202500000,ACC-93810,SMB,Wales,email,Reporting,service_outage,Critical,2025-09-06 18:51:00,Saturday,...,Amelia,6,13.11,62,0,2025-09-07 20:56:43,1,26.10,Critical,1
1,TCK-202500001,ACC-82357,SMB,East Midlands,web,Billing,data_quality,Medium,2026-02-27 08:29:00,Friday,...,Isla,30,8.35,289,0,2026-02-28 05:55:43,0,21.45,Medium,0
2,TCK-202500002,ACC-45093,SMB,East Midlands,web,Api Integration,access_request,Medium,2025-10-19 05:16:00,Sunday,...,Muhammad,19,11.28,215,0,2025-10-19 17:38:32,0,12.38,Medium,0
3,TCK-202500003,ACC-44973,Enterprise,Northern Ireland,web,Workflow Automation,how_to,Critical,2025-12-01 03:05:00,Monday,...,Isla,25,20.45,261,0,2025-12-01 10:42:39,1,7.63,Critical,1
4,TCK-202500004,ACC-26483,SMB,South West,email,Security,performance,High,2026-01-16 11:48:00,Friday,...,Harry,35,0.06,139,0,2026-01-16 23:11:33,0,11.39,High,0


## Feature Engineering Strategy

Feature engineering will focus on creating informative variables that are available at the time a support ticket is submitted. To ensure realistic model deployment, only information known at the prediction point will be retained.

Potential leakage variables such as `resolved_at`, `resolution_hours`, `final_priority`, and `escalated_prior_to_resolution` will be excluded because they contain information generated after ticket handling has begun.

Feature transformations will remain simple, interpretable, and business-focused. Examples include extracting temporal information from submission timestamps, creating weekend indicators, and preparing numerical variables for modelling.

The resulting dataset will be saved in a modelling-ready format for use in subsequent training and evaluation notebooks.


## Feature engineering rules

- Use only columns available at the chosen prediction point.
- Exclude leakage columns from the primary classifier.
- Keep transformations simple and explainable.
- Save a modelling-ready sample for later notebooks.

In [3]:
features = df.copy()
features["submitted_at"] = pd.to_datetime(features["submitted_at"], errors="coerce")
features["submitted_hour"] = features["submitted_at"].dt.hour
features["submitted_dayofweek_num"] = features["submitted_at"].dt.dayofweek
features["submitted_month"] = features["submitted_at"].dt.month
features["is_weekend"] = features["submitted_dayofweek_num"].isin([5, 6]).astype(int)

features["contract_value_log1p"] = np.log1p(features["contract_value_gbp"].clip(lower=0))
features["queue_pressure"] = features["agent_queue_length_at_submit"] / (features["agent_experience_months"] + 1)
features["negative_sentiment_flag"] = (features["avg_sentiment_score"] < -0.35).astype(int)
features["high_backlog_flag"] = (features["backlog_age_hours"] > features["backlog_age_hours"].quantile(0.80)).astype(int)

# Simple text-derived features
features["message_word_count"] = features["customer_message"].fillna("").str.split().str.len()
features["message_has_deadline"] = features["customer_message"].fillna("").str.lower().str.contains("deadline|asap|urgent|critical|blocked").astype(int)

# Leakage-risk columns for the intake classifier
leakage_columns = ["resolved_at", "escalated_prior_to_resolution", "resolution_hours", "final_priority"]
id_columns = ["ticket_id", "account_id", "customer_message"]

target = "sla_breached"
feature_cols = [c for c in features.columns if c not in leakage_columns + id_columns + [target]]

print("Number of candidate features:", len(feature_cols))
print(feature_cols)


Number of candidate features: 32
['customer_segment', 'uk_region', 'support_channel', 'product_area', 'issue_category', 'priority_initial', 'submitted_at', 'day_of_week', 'hour_of_day', 'customer_tenure_months', 'contract_value_gbp', 'previous_tickets_90d', 'avg_sentiment_score', 'message_length', 'contains_urgent_keyword', 'contains_refund_keyword', 'agent_queue_length_at_submit', 'assigned_agent', 'agent_experience_months', 'backlog_age_hours', 'first_response_minutes', 'reopened_last_90d', 'submitted_hour', 'submitted_dayofweek_num', 'submitted_month', 'is_weekend', 'contract_value_log1p', 'queue_pressure', 'negative_sentiment_flag', 'high_backlog_flag', 'message_word_count', 'message_has_deadline']


In [4]:
feature_cols = [c for c in feature_cols if c != "submitted_at"]

In [6]:
print("Number of candidate features:", len(feature_cols))

Number of candidate features: 31


### Feature Engineering Validation

A total of 31 candidate features were retained for modelling after removing identifier fields and known leakage variables. Temporal, operational, customer, sentiment, and workload-related features were successfully engineered to provide additional predictive information while maintaining business interpretability.

The final feature set contains only information available at ticket intake, ensuring that subsequent model evaluation reflects realistic deployment conditions.


## Save modelling-ready dataset

In [9]:
model_df = features[feature_cols + [target, "resolution_hours"]].copy()

OUTPUT_PATH = Path("../01_Data/processed data/model_ready_support_sla_sample.csv")

model_df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)

display(model_df.head())

Saved: ..\01_Data\processed data\model_ready_support_sla_sample.csv


,customer_segment,uk_region,support_channel,product_area,issue_category,priority_initial,day_of_week,hour_of_day,customer_tenure_months,contract_value_gbp,...,submitted_month,is_weekend,contract_value_log1p,queue_pressure,negative_sentiment_flag,high_backlog_flag,message_word_count,message_has_deadline,sla_breached,resolution_hours
0,SMB,Wales,email,Reporting,service_outage,Critical,Saturday,18,40.0,7522.08,...,9.0,1,8.925731,0.714286,0,0,15,1,1,26.10
1,SMB,East Midlands,web,Billing,data_quality,Medium,Friday,8,39.0,5961.51,...,2.0,0,8.693247,1.064516,0,0,13,0,0,21.45
2,SMB,East Midlands,web,Api Integration,access_request,Medium,Sunday,5,27.0,12098.44,...,10.0,1,9.400914,1.900000,0,0,17,0,0,12.38
3,Enterprise,Northern Ireland,web,Workflow Automation,how_to,Critical,Monday,3,29.0,23198.54,...,12.0,0,10.051888,0.884615,1,1,16,1,1,7.63
4,SMB,South West,email,Security,performance,High,Friday,11,26.0,15328.55,...,1.0,0,9.637538,0.833333,0,0,14,1,0,11.39


| Feature Group                 | Examples                                                                                                                                                   | Reason for Inclusion                                                                                                           |
| ----------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| Customer Attributes           | `customer_segment`, `customer_tenure_months`, `contract_value_gbp`, `contract_value_log1p`                                                                 | Capture customer importance, account maturity, and business value which may influence support prioritisation and SLA outcomes. |
| Ticket Characteristics        | `issue_category`, `priority_initial`, `message_length`, `message_word_count`, `contains_urgent_keyword`, `contains_refund_keyword`, `message_has_deadline` | Represent ticket complexity, urgency, and the nature of the support request.                                                   |
| Operational Workload          | `agent_queue_length_at_submit`, `backlog_age_hours`, `queue_pressure`, `high_backlog_flag`                                                                 | Reflect support team workload and operational pressure at ticket submission time.                                              |
| Agent Factors                 | `assigned_agent`, `agent_experience_months`                                                                                                                | Capture differences in agent capacity, expertise, and ticket handling performance.                                             |
| Historical Customer Behaviour | `previous_tickets_90d`, `reopened_last_90d`                                                                                                                | Measure previous support activity and recurring customer issues.                                                               |
| Sentiment Features            | `avg_sentiment_score`, `negative_sentiment_flag`                                                                                                           | Capture customer frustration and potential escalation risk.                                                                    |
| Temporal Features             | `submitted_hour`, `submitted_dayofweek_num`, `submitted_month`, `is_weekend`, `day_of_week`                                                                | Represent seasonal, weekly, and hourly operational patterns affecting SLA performance.                                         |
| Support Context               | `support_channel`, `product_area`, `uk_region`                                                                                                             | Capture differences in support workflows, products, and geographic demand patterns.                                            |


| Excluded Feature                | Reason for Exclusion                                                                                                                                   |
| ------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `ticket_id`                     | Unique ticket identifier with no predictive value for SLA breach prediction.                                                                           |
| `account_id`                    | Customer account identifier; serves only as a reference and does not provide meaningful predictive information.                                        |
| `customer_message`              | Raw text field excluded from the baseline model and replaced with engineered text features such as message length, word count, and urgency indicators. |
| `submitted_at`                  | Original timestamp replaced by engineered temporal features including submission hour, day of week, month, and weekend indicators.                     |
| `resolved_at`                   | Not available at ticket creation; contains post-resolution information and would introduce target leakage.                                             |
| `resolution_hours`              | Directly reflects the final ticket outcome and would provide future information unavailable at prediction time.                                        |
| `final_priority`                | Represents the final ticket priority after processing and may contain information unavailable at initial ticket submission.                            |
| `escalated_prior_to_resolution` | Indicates escalation activity occurring after ticket creation and before resolution, creating target leakage risk.                                     |


### Short Summary 
Feature selection focused on variables available at the time of ticket submission. Customer, operational, temporal, sentiment, and ticket-related features were retained because they provide meaningful information about SLA breach risk. Known leakage variables and identifier fields were excluded to ensure realistic model performance and prevent information from future events influencing predictions.